# Scrape Radio Radicale

Given a URL:
1. download audio
2. get timestamps and speech date
3. trim audio to timestamps
4. text to speech
5. save into csv 
   1. politician name
   2. date of speech
   3. metadata
      1. url
      2. others
   4. transcribed text

In [ ]:
POLITICIANS = [
    "draghi",
    "meloni",
    "conte",
    "gentiloni",
    "renzi",
    "letta",
    "monti",
    "dini",
    "prodi",
    "d'alema",
    "ciampi",
    "berlusconi",
    "amato",
    "de mita",
    "craxi",
    "goria",
    "spadolini",
    "forlani",
    "cossiga",
    "rumor",
    "emilio",
    "andreotti",
    "moro",
    "leone",
    "segni",
    "scelba"
]

SUBJECTS = [
    "/soggetti/46315/mario-draghi",
    "/soggetti/94729/giorgia-meloni",
    "/soggetti/110925/giuseppe-conte",
    "/soggetti/243521/paolo-gentiloni",
    "/soggetti/74722/matteo-renzi",
    "/soggetti/40755/enrico-letta",
    "/soggetti/35269/mario-monti",
    "/soggetti/24012/lamberto-dini",
    "/soggetti/6661/romano-prodi",
    "/soggetti/4773/massimo-d-alema",
    "/soggetti/17588/carlo-azeglio-ciampi",
    "/soggetti/11165/silvio-berlusconi",
    "/soggetti/311/giuliano-amato",
    "/soggetti/1781/ciriaco-de-mita",
    "/soggetti/286/bettino-craxi",
    "/soggetti/831/giovanni-goria",
    "/soggetti/253/giovanni-spadolini",
    "/soggetti/1043/arnaldo-forlani",
    "/soggetti/595/francesco-cossiga",
    "/soggetti/1769/mariano-rumor",
    "/soggetti/6010/colombo-emilio",
    "/soggetti/99/giulio-andreotti",
    "/soggetti/1325/aldo-moro",
    "/soggetti/6656/giovanni-leone",
    "/soggetti/101437/antonio-segni",
    "/soggetti/191965/mario-scelba"
]


In [2]:
import os
from pathlib import Path

testing = False
politician = "andreotti"
model_name  = "medium"      # tiny, base, medium, large
lang_code   = "Italian"
is_clock_time = True     # timestamps are either in hh:mm starting from 00:00 or represent actual time of the day 
                            
# url scraping
BASE_URL = "https://www.radioradicale.it"
SUBJECT_URL = f"{BASE_URL}/soggetti/99/giulio-andreotti"

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # go to ~/crossdem/ by jumping up twice
OUT_DIR = os.path.join(BASE_DIR, f"datasets/{politician}")
AUDIO_DIR = os.path.join(OUT_DIR, f"audio_out")
CSV_DIR = os.path.join(OUT_DIR, f"csv_out")

if testing:
    OUT_DIR = os.path.join(BASE_DIR, "notebooks/out")
    AUDIO_DIR = OUT_DIR
    CSV_DIR = OUT_DIR

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

## Download audio
Given a radio radicale page url, extract the audio embedding url and downloads it. Also embed metadata in the mp3 file.

In [3]:
import re
import json
import requests
import yt_dlp
import subprocess


def get_audio_url(page_url: str) -> str:
    resp = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    match = re.search(r'rtsp://[^"]+\.mp3', resp.text)
    if not match:
        raise ValueError("No rtsp audio link found on the page")
    return match.group(0)

def sanitize_url(page_url: str) -> str:
    """
    go from https://www.radioradicale.it/scheda/000000/somethingsomethign
    to      https://www.radioradicale.it/scheda/000000
    """
    base = page_url.split("?")[0]
    parts = base.split("/")
    return "/".join(parts[:5])  # https: + '' + domain + scheda + ID

def extract_id_from_url(url):
    """Extract the scheda ID from a Radio Radicale URL."""
    # e.g. /scheda/58768/... → "58768"
    parts = url.split("/scheda/")
    if len(parts) > 1:
        return parts[1].split("/")[0]
    return None

def download_audio_subprocess(url: str, out_dir: str, politician: str = "politician") -> dict:
    """
    Alternative to download_audio that does not struggle with long videos.
    """
    import glob, os
    url = sanitize_url(url)
    id = extract_id_from_url(url)
    stem = f"{id}_{politician}"
    final_file = f"{out_dir}/{stem}.mp3"

    print(f"Downloading {url}")
    print(f"Resulting file {final_file}")

    result = subprocess.run([
        "yt-dlp",
        "-x",
        "--audio-format", "mp3",
        "--audio-quality", "0",
        #"--sleep-requests", "2",
        #"--sleep-interval", "5",
        #"--max-sleep-interval", "15",
        #"--limit-rate", "5M",
        "-o", f"{out_dir}/{stem}_part%(playlist_index)s.%(ext)s",
        "--concat-playlist", "always",
        "-o", f"pl_video:{out_dir}/{stem}_part%(playlist_index)s.%(ext)s",
        url
    ], capture_output=False, text=True)



    if result.returncode != 0:
        raise RuntimeError(f"yt-dlp exited {result.returncode}")
    print("Concatenate fragments")
    ## ---------- concatenate audio fragments -------------------------
    # yt-dlp concat destination is always part0 (it overwrites the first part in-place)
    concat_result = f"{out_dir}/{stem}_part0.mp3"
    if not os.path.exists(concat_result):
        # fallback: maybe it was a single-part download with no index
        candidates = glob.glob(f"{out_dir}/{stem}*.mp3")
        if not candidates:
            raise FileNotFoundError(f"No mp3 found in {out_dir} for stem {stem}")
        concat_result = candidates[0]

    os.rename(concat_result, final_file)
    print("Fetch metadata")
        # ── 3. Fetch rich metadata (no re-download) ───────────────────────────────
    meta_result = subprocess.run([
        "yt-dlp",
        "--dump-json",
        "--no-playlist",       # get the page-level info, not per-entry
        "--flat-playlist",     # don't resolve individual entries
        url,
    ], capture_output=True, text=True)

    if meta_result.returncode == 0 and meta_result.stdout.strip():
        # --dump-json may emit one JSON object per line for playlists; take the last
        # (the playlist container) or first (single video) depending on structure
        lines = [l for l in meta_result.stdout.splitlines() if l.strip()]
        try:
            info = json.loads(lines[-1])
        except json.JSONDecodeError:
            info = {}
    else:
        info = {}

    # ── 4. Normalise to the same shape as download_audio's sanitize_info ──────
    info["filename"] = os.path.basename(final_file)
    info["file_id"] = id
    info.setdefault("url", url)
    return info

    return {"filename": final_file, "url": url}

## Testing -- just call download_audio() in the run() afterwards 
if __name__ == "__main__" and True:
    PAGE_URL = "https://www.radioradicale.it/scheda/268754"
    test_metadata = download_audio_subprocess(PAGE_URL, AUDIO_DIR)
    for a,b in test_metadata.items():
        print(f"{a:<15} {b}")

Resulting file /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/268754_politician.mp3


         It is strongly recommended to always use the latest version.
         You installed yt-dlp from a manual build or with a package manager; Use that to update.
         To suppress this warning, add --no-update to your command/config.


[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/268754
[RadioRadicale] 268754: Downloading webpage
[RadioRadicale] 268754-0: Downloading m3u8 information
[info] 268754: Downloading 1 format(s): 82
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 9
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/268754_politician_partNA.m4a
[download] 100% of  846.92KiB in 00:00:01 at 570.49KiB/s               
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/268754_politician_partNA.m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/268754_politician_partNA.mp3
Deleting original file /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/268754_politician_partNA.m4a (pass -k to keep)
Concatenate fragments
Fetch metadata
id              268754
title           La Questione morale e le polemiche s

All video metadata all extracted and embedded in the mp3 file 

## Extract speech date & timestamps
Uses BeautifulSup to parse all XML data and try to find out when a speech was made

In [4]:
import re
from datetime import date

import requests
from bs4 import BeautifulSoup


# ── HTTP ────────────────────────────────────────────────────────────────────

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "it-IT,it;q=0.9,en;q=0.5",
}

def fetch_page(url: str, timeout: int = 20) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ── Date patterns ───────────────────────────────────────────────────────────

MONTHS_IT = {
    "gennaio": 1, "febbraio": 2, "marzo": 3, "aprile": 4,
    "maggio": 5, "giugno": 6, "luglio": 7, "agosto": 8,
    "settembre": 9, "ottobre": 10, "novembre": 11, "dicembre": 12,
}
MONTHS_IT_SHORT = {
    "gen": 1, "feb": 2, "mar": 3, "apr": 4, "mag": 5, "giu": 6,
    "lug": 7, "ago": 8, "set": 9, "ott": 10, "nov": 11, "dic": 12,
}

RE_DATE_LONG   = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT) + r")\s+(\d{4})\b", re.I)
RE_DATE_SHORT  = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT_SHORT) + r")\s+(\d{4})\b", re.I)
RE_DATE_DOTTED = re.compile(r"\b(\d{1,2})\.(\d{2})\.(\d{4})\b")
RE_DATE_ISO    = re.compile(r"(\d{4})-(\d{2})-(\d{2})")

def _try_long(text):   m = RE_DATE_LONG.search(text);   return date(int(m[3]), MONTHS_IT[m[2].lower()],       int(m[1])) if m else None
def _try_short(text):  m = RE_DATE_SHORT.search(text);  return date(int(m[3]), MONTHS_IT_SHORT[m[2].lower()], int(m[1])) if m else None
def _try_dotted(text): m = RE_DATE_DOTTED.search(text); return date(int(m[3]), int(m[2]),                      int(m[1])) if m else None
def _try_iso(text):    m = RE_DATE_ISO.search(text);    return date(int(m[1]), int(m[2]),                      int(m[3])) if m else None

def find_date(text: str) -> date | None:
    return _try_long(text) or _try_short(text) or _try_dotted(text) or _try_iso(text)


# ── Extract location ─────────────────────────────────────────────────────────

RE_LOCATION = re.compile(r'-\s+([A-ZÀÈÉÌÒÙ][A-ZÀÈÉÌÒÙ\s]+?)\s+-\s+\d{2}:\d{2}')

def extract_location(soup: BeautifulSoup) -> str | None:
    tag = soup.find("div", class_="primo_suffisso")
    if not tag:
        return None
    m = RE_LOCATION.search(tag.get_text())
    return m.group(1).strip() if m else None


# ── Timestamp parsing ───────────────────────────────────────────────────────

RE_DURATA = re.compile(r'(\d+:\d+)\s+Durata:\s+(\d+)\s+min', re.I)
RE_INT_ID = re.compile(r'^int(\d+)$')
RE_D_SEC  = re.compile(r'^d(\d+)$')

def parse_timestamps(li, is_clock_time: bool = False, event_start: str | None = None) -> dict:
    result = {"int_id": None, "start_time": None, "duration_min": None, "duration_seconds": None}

    print(f"parse timestamps")
    print(f"is clock time = {is_clock_time}")
    print(f"event start = {event_start}")
    for cls in li.get("class", []):
        if m := RE_INT_ID.match(cls):
            result["int_id"] = int(m.group(1))
        if m := RE_D_SEC.match(cls):
            result["duration_seconds"] = int(m.group(1))

    durata_div = li.find("div", class_="durata")
    if durata_div:
        if m := RE_DURATA.search(durata_div.get_text()):
            raw_time = m.group(1)
            result["duration_min"] = int(m.group(2))
            
            if is_clock_time and event_start:
                h0, m0 = (int(x) for x in event_start.split(":"))
                h1, m1 = (int(x) for x in raw_time.split(":"))
                offset_min = (h1 * 60 + m1) - (h0 * 60 + m0)
                result["start_time"] = f"{offset_min // 60:02d}:{offset_min % 60:02d}"
            else:
                result["start_time"] = raw_time

    return result


# ── Date extraction ─────────────────────────────────────────────────────────

def extract_page_date(soup: BeautifulSoup) -> tuple[date | None, str]:
    """Page-level event date (meta > title > stamp > sommario prose)."""
    for attr, val in [("name", "dcterms.date"), ("property", "article:published_time")]:
        tag = soup.find("meta", {attr: val})
        if tag and tag.get("content"):
            d = _try_iso(tag["content"])
            if d: return d, "meta_tag"

    title = soup.find("title")
    if title:
        d = _try_dotted(title.get_text())
        if d: return d, "page_title"

    page_text = soup.get_text(" ", strip=True)
    d = _try_short(page_text)
    if d: return d, "page_stamp"

    d = _try_long(page_text)
    if d: return d, "page_sommario"

    return None, "not_found"


def extract_speaker_date(soup: BeautifulSoup, speaker: str) -> tuple[date | None, str]:
    """
    Per-speaker date, checked in priority order:
      1. The speaker's own int_subtext
      2. The preceding li's int_subtext (introduction pattern,
         e.g. 'L\'on. Fanfani a una conferenza stampa del 22 marzo 1962')
      3. Any date anywhere in the speaker's <li> block
    Only uses the FIRST occurrence of the speaker for date purposes.
    """
    items = [li for li in soup.select("li.intervento") if li.find("h2")]

    for idx, li in enumerate(items):
        if speaker.lower() not in li.find("h2").get_text().lower():
            continue

        subtext = li.find("div", class_="int_subtext")
        if subtext:
            d = find_date(subtext.get_text())
            if d: return d, "speaker_subtext"

        if idx > 0:
            prev_subtext = items[idx - 1].find("div", class_="int_subtext")
            if prev_subtext:
                text = prev_subtext.get_text()
                if speaker.lower() in text.lower():
                    d = find_date(text)
                    if d: return d, "preceding_subtext"

        d = find_date(li.get_text(" ", strip=True))
        if d: return d, "speaker_block"

        return None, "speaker_block_no_date"

    return None, "speaker_not_found"


# ── Main ────────────────────────────────────────────────────────────────────


def extract_speech_details(url: str, speaker: str, is_clock_time: bool = False, timeout: int = 20) -> dict:
    result = dict(
        url=url, speaker_query=speaker, speaker_found=False,
        speech_date=None, date_source=None, page_event_date=None,
        interventions=[],
        error=None,
    )
    try:
        soup = fetch_page(url, timeout=timeout)
    except Exception as e:
        result["error"] = str(e)
        return result

    page_date, page_src = extract_page_date(soup)
    if page_date:
        result["page_event_date"] = page_date.isoformat()

    location = extract_location(soup)
    result["location"] = location if (location is not None and location != "RADIO") else "UNKNOWN"

    # Extract event_start from the first intervento on the page
    event_start = None
    first_li = soup.select_one("li.intervento")
    if first_li:
        durata_div = first_li.find("div", class_="durata")
        if durata_div:
            m = RE_DURATA.search(durata_div.get_text())
            if m:
                event_start = m.group(1)

    # Collect ALL occurrences of the speaker
    for li in soup.select("li.intervento"):
        h2 = li.find("h2")
        if h2 and speaker.lower() in h2.get_text().lower():
            result["speaker_found"] = True
            ts = parse_timestamps(li, is_clock_time=is_clock_time, event_start=event_start)
            if ts["start_time"] or ts["duration_seconds"] is not None:
                result["interventions"].append(ts)

    if not result["speaker_found"]:
        result["error"] = f"'{speaker}' not found in interventi"
        result["date_source"] = "llm_needed"
        return result

    if not result["interventions"]:
        result["interventions"] = "no timestamps"

    spk_date, spk_src = extract_speaker_date(soup, speaker)
    if spk_date:
        result["speech_date"] = spk_date.isoformat()
        result["date_source"] = spk_src
    elif page_date:
        result["speech_date"] = page_date.isoformat()
        result["date_source"] = f"page_event ({page_src})"
    else:
        result["date_source"] = "llm_needed"

    return result



## Cut audio timestamps 
implement into the pipeline

In [5]:
import subprocess
from pathlib import Path

def trim_to_speaker(audio_filename: str, interventions: list) -> Path:
    """
    Given a list of interventions (from extract()), concatenate only the
    speaker's segments back-to-back and overwrite the original file.
    
    audio_filename must be a string like "85823_politician.mp3"
    interventions is r["interventions"] — list of dicts with start_time (h:mm)
    and duration_seconds.
    """
    print(f"trim_to_speaker(audio_filename = {audio_filename}, interventions: list) -> Path:")
    input_path = Path(AUDIO_DIR) / audio_filename

    def hhmm_to_seconds(t: str) -> int:
        h, m = t.split(":")
        return int(h) * 3600 + int(m) * 60

    # Build one -ss/-t segment filter per intervention
    # Using ffmpeg's trim+adelay approach is complex; simpler: extract each
    # segment to a temp file, then concatenate with concat demuxer
    tmp_files = []
    for i, iv in enumerate(interventions):
        tmp = input_path.with_name(f"_tmp_{i}_{input_path.name}")
        start_sec = hhmm_to_seconds(iv["start_time"])
        cmd = [
            "ffmpeg", "-y",
            "-ss", str(start_sec),
            "-i", str(input_path),
            "-t", str(iv["duration_seconds"]),
            "-acodec", "copy",
            str(tmp),
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        tmp_files.append(tmp)

    # Write concat list file
    concat_list = input_path.with_name("_concat_list.txt")
    concat_list.write_text("\n".join(f"file '{f.name}'" for f in tmp_files))

    # Concatenate into a temp output, then overwrite original
    tmp_out = input_path.with_name(f"_out_{input_path.name}")
    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(concat_list),
        "-acodec", "copy",
        str(tmp_out),
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Cleanup temps, overwrite original
    for f in tmp_files:
        f.unlink(missing_ok=True)
    concat_list.unlink(missing_ok=True)
    if False:                               # replace or create a new one
        trimmed_path = input_path.with_stem(input_path.stem + "_trimmed")   
        tmp_out.replace(trimmed_path)
    else:
        tmp_out.replace(input_path)

    return input_path

## Text to speech 
Take trimmed speeches and transcribe them into text

In [6]:
import json
import os
import subprocess
import csv
from datetime import datetime
from pathlib import Path

In [7]:

# ──────────────────────────────────────────────────────────────────────────────
def speech_to_text(audio_metadata, speech_details, audio_path, politician):


    historical_date = speech_details["speech_date"]
    location = speech_details["location"]
    title = audio_metadata["fulltitle"]
    file_id = audio_metadata["file_id"]
    filename = audio_metadata["filename"]


    output_dir  = str(Path(OUT_DIR) / f"output-{model_name}")
    audio_file  = str(Path(AUDIO_DIR) / filename)
    csv_output  = str(Path(CSV_DIR) / f"{file_id}_{politician}_s2t.csv")


    # ──────────────────────────────────────────────────────────────────────────────
    # 3. Transcribe with Whisper
    if not os.path.exists(audio_file):
        raise FileNotFoundError(f"Audio file '{audio_file}' not found.")

    print("Start transcription...")
    subprocess.run([
        "whisper",
        "--language",        lang_code,
        "--word_timestamps", "True",
        "--model",           model_name,
        "--output_dir",      output_dir,
        "--device",          "cuda",
        audio_file
    ], check=True)

    # ──────────────────────────────────────────────────────────────────────────────
    # 4. Read the plain-text transcript (.txt produced by Whisper)
    #transcript_path = os.path.join(output_dir, os.path.splitext(audio_file)[0] + ".txt")

    print("Transcribed")
    audio_path = Path(audio_file)
    transcript_path = Path(output_dir) / f"{audio_path.stem}.txt"

    # If you need it as a string for the open() function:
    transcript_path = str(transcript_path)
    if not os.path.exists(transcript_path):
        raise FileNotFoundError(
            f"Transcript not found at '{transcript_path}'. "
            "Check the output_dir and audio filename."
        )

    with open(transcript_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # ──────────────────────────────────────────────────────────────────────────────
    # 5. Write CSV
    print("Writing into csv...")
    audio_path_str = str(audio_path)

    row = {
        "politician":      f"{politician}",
        "historical_date": datetime.strptime(historical_date, "%Y-%m-%d").date() if historical_date else "",
        "location":        location,
        "title":           title.replace("\n", " ").replace("\r", " ").strip(),
        "url":             url,
        "audio_file":      audio_path_str[audio_path_str.index("crossdem"):],
        "text":            text.replace("\n", " ").replace("\r", " ").strip(),
    }

    write_header = not os.path.exists(csv_output)
    with open(csv_output, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    print(f"Done! Row appended to '{csv_output}'")

## Scrape all URLs automatically

In [8]:
"""
Radio Radicale – Poitician audio URL scraper.

Main entry point:
    urls = get_all_politician_audio_urls()   # returns list[str]
"""

import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


# Category filter values to scrape (skipping 1=Istituzioni and All)
CATEGORIES = {
    "Dibattiti":     2,
    "Rubriche":      3,
    "Interviste":    4,
    "Manifestazioni":5,
    "Processi":      6,
    "Partiti":       7,
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}


def _scrape_category(session: requests.Session, cat_value: int) -> list[str]:
    """Scrape all audio scheda URLs for one category, following pagination."""
    urls: list[str] = []
    page = 0

    while True:
        params = {"field_registrazione_raggruppamenti_radio": cat_value, "page": page}
        resp = session.get(SUBJECT_URL, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")

        # Each audio entry: <li class="views-row ...">
        #   <div class="ls_text"><h3><a href="/scheda/...?i=...">
        for li in soup.select("ol.lista_list li.views-row"):
            # Only rows that have audio
            if not li.select_one("div.tipo_media.audio"):
                continue
            a = li.select_one("div.ls_text h3 a")
            if a and a.get("href"):
                full_url = urljoin(BASE_URL, a["href"])
                urls.append(full_url)

        # Check for a "next page" link
        next_link = soup.select_one("li.pager__item--next a")
        if not next_link:
            break

        page += 1
        time.sleep(0.5)   # be polite

    return urls


def get_all_audio_urls(
    categories: dict[str, int] | None = None,
    verbose: bool = True,
) -> list[str]:
    """
    Scrape all Politician audio scheda URLs from Radio Radicale.

    Parameters
    ----------
    categories : dict mapping label → filter value (defaults to the six
                 non-Istituzioni categories defined at module level).
    verbose    : print progress info.

    Returns
    -------
    Deduplicated list of absolute scheda URLs, e.g.
    ['https://www.radioradicale.it/scheda/55884/...?i=2447650', ...]
    """
    if categories is None:
        categories = CATEGORIES

    session = requests.Session()
    all_urls: list[str] = []
    seen_paths: set[str] = set()   # keyed on URL path, ignoring ?i=

    for label, value in categories.items():
        if verbose:
            print(f"  Scraping category: {label} (value={value}) …", flush=True)
        cat_urls = _scrape_category(session, value)
        new = []
        for u in cat_urls:
            path = u.split("?")[0]
            if path not in seen_paths:
                seen_paths.add(path)
                new.append(u)
        all_urls.extend(new)
        if verbose:
            print(f"    → {len(cat_urls)} found, {len(new)} new (total so far: {len(all_urls)})")

    if verbose:
        print(f"\nDone. {len(all_urls)} unique audio URLs collected.")

    return all_urls


# ── quick smoke-test ──────────────────────────────────────────────────────────
if __name__ == "__main__" and False:
    urls = get_all_audio_urls(verbose=True)
    for u in urls:
        print(u)

## Run
run all together, save the result as a separate csv file for each video

In [9]:
# ── Run ─────────────────────────────────────────────────────────────────────
#urls = ["https://www.radioradicale.it/scheda/265208/andreotti-la-vita-di-un-uomo-politico-la-storia-di-unepoca?i=735960"]
import glob
from time import sleep

urls = get_all_audio_urls()

def get_processed_ids(output_dir=AUDIO_DIR):
    processed = set()
    for path in glob.glob(os.path.join(output_dir, "*_*.mp3")):
        stem = os.path.splitext(os.path.basename(path))[0]
        scheda_id = stem.split("_")[0]
        processed.add(scheda_id)
    return processed

def extract_id_from_url(url):
    """Extract the scheda ID from a Radio Radicale URL."""
    # e.g. /scheda/58768/... → "58768"
    parts = url.split("/scheda/")
    if len(parts) > 1:
        return parts[1].split("/")[0]
    return None

for url in urls:
    """
    One audio at a time, extract date and timestamps, then download, then trim it based on extracted timestamps, 
    then transcribe it with OpenAI Whisper and finally save it in a unique CSV file.
    """
    processed_ids = get_processed_ids()
    scheda_id = extract_id_from_url(url)
    if scheda_id in processed_ids:
        print(f"Skipping {scheda_id}, already processed.")
        continue

    print(f"Processing {url}")
    speech_details = extract_speech_details(url, speaker=politician, is_clock_time=is_clock_time)
    for key, value in speech_details.items():
        print(f"{key:<18} {value}")

    speech_date = speech_details["speech_date"]
    timestamps = speech_details["interventions"]
    location = speech_details["location"]

    # Download audio
    audio_metadata = None
    if any(i['start_time'] is None for i in timestamps):
        print("No timestamps, skip")
    else:
        audio_metadata = download_audio_subprocess(url, AUDIO_DIR, politician)
    
    # Trim audio based on timestamps
    if any(i['start_time'] is None for i in timestamps):
        print("No timestamps, skip")
    else:
        sleep(1)
        audio_path = trim_to_speaker(audio_metadata["filename"], timestamps)
        speech_to_text(audio_metadata, speech_details, audio_path, politician)


  Scraping category: Dibattiti (value=2) …
    → 710 found, 326 new (total so far: 326)
  Scraping category: Rubriche (value=3) …
    → 14 found, 12 new (total so far: 338)
  Scraping category: Interviste (value=4) …
    → 168 found, 166 new (total so far: 504)
  Scraping category: Manifestazioni (value=5) …
    → 41 found, 30 new (total so far: 534)
  Scraping category: Processi (value=6) …
    → 49 found, 24 new (total so far: 558)
  Scraping category: Partiti (value=7) …
    → 29 found, 5 new (total so far: 563)

Done. 563 unique audio URLs collected.
Processing https://www.radioradicale.it/scheda/278642/celebrazione-del-60deg-anniversario-dellalleanza-atlantica?i=690689
parse timestamps
is clock time = True
event start = 9:50
url                https://www.radioradicale.it/scheda/278642/celebrazione-del-60deg-anniversario-dellalleanza-atlantica?i=690689
speaker_query      andreotti
speaker_found      True
speech_date        2009-05-11
date_source        page_event (meta_tag)
page_e

         It is strongly recommended to always use the latest version.
         You installed yt-dlp from a manual build or with a package manager; Use that to update.
         To suppress this warning, add --no-update to your command/config.


[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/278642
[RadioRadicale] 278642: Downloading webpage
[RadioRadicale] 278642-0: Downloading m3u8 information
[RadioRadicale] 278642-1: Downloading m3u8 information
[download] Downloading multi_video: Celebrazione del 60° anniversario dell'Alleanza Atlantica
[RadioRadicale] Playlist Celebrazione del 60° anniversario dell'Alleanza Atlantica: Downloading 2 items of 2
[download] Downloading item 1 of 2
[info] 278642-0: Downloading 1 format(s): 82
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 964
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/278642_andreotti_part1.m4a
[download] 100% of   98.08MiB in 00:01:34 at 1.04MiB/s                   
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/278642_andreotti_part1.m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andr

         It is strongly recommended to always use the latest version.
         You installed yt-dlp from a manual build or with a package manager; Use that to update.
         To suppress this warning, add --no-update to your command/config.


[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/271950
[RadioRadicale] 271950: Downloading webpage
[RadioRadicale] 271950-0: Downloading m3u8 information
[info] 271950: Downloading 1 format(s): 82
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 700
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/271950_andreotti_partNA.m4a
[download] 100% of   71.54MiB in 00:01:05 at 1.09MiB/s                   
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/271950_andreotti_partNA.m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/271950_andreotti_partNA.mp3
Deleting original file /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/271950_andreotti_partNA.m4a (pass -k to keep)
Concatenate fragments
Fetch metadata
trim_to_speaker(audio_filename = 271950_andreotti.mp3, interventions: list)

         It is strongly recommended to always use the latest version.
         You installed yt-dlp from a manual build or with a package manager; Use that to update.
         To suppress this warning, add --no-update to your command/config.


[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/267577
[RadioRadicale] 267577: Downloading webpage
[RadioRadicale] 267577-0: Downloading m3u8 information
[RadioRadicale] 267577-1: Downloading m3u8 information
[download] Downloading multi_video: Il trattato di Lisbona e l'avvenire dell'Europa
[RadioRadicale] Playlist Il trattato di Lisbona e l'avvenire dell'Europa: Downloading 2 items of 2
[download] Downloading item 1 of 2
[info] 267577-0: Downloading 1 format(s): 82
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1050
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/267577_andreotti_part1.m4a
[download] 100% of  107.12MiB in 00:01:40 at 1.07MiB/s                     
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/267577_andreotti_part1.m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/datasets/andreotti/audio_out/2


ERROR: Interrupted by user


KeyboardInterrupt: 